In [20]:
from langchain.chat_models import init_chat_model
from langchain_text_splitters import MarkdownHeaderTextSplitter

# =============1.准备文档================
# 1.读取文档
with open('./resources/评估.md', 'r', encoding='utf-8') as f:
    doc = f.read()

# 2.切分文档
headers_to_split_on = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on,
    strip_headers=False # 是否去除文本中的标题，False：不去除
)
# list[Document]
documents = markdown_splitter.split_text(doc)
print(len(documents))

21


In [21]:
for i, document in enumerate(documents, start=1):
    document.id = f"doc_{i}"

In [28]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# =============2.准备VectorStore================

# 1.初始化向量模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 向量维度，可选：减少维度以节省存储
)

# 2.初始化VectorStore
vector_store = InMemoryVectorStore(ollama_embeddings)

In [31]:
# =============3.添加和删除文档================

ids = vector_store.add_documents(documents)

In [30]:
vector_store.delete(["doc_1"])

In [25]:
# 结果是一个list[Document]
result = vector_store.similarity_search(
    query="论语中教育的目的是什么",
    k=2
)

[Document(id='doc_3', metadata={'H1': '第一章 教育概述', 'H2': '第一节 中外教育家及其教育思想', 'H3': '（三）孔子及《论语》主要思想'}, page_content='### （三）孔子及《论语》主要思想  \n**教育作用**：庶、富、教；性相近，习相远。  \n**教育对象**：“有教无类”，教育民主思想。  \n**教育目的**：以完善人格为教育的首要目的，培养士和君子。  \n**教学内容**：文、行、忠、义。  \n**教学过程**：学、思、习、行。  \n**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  \n**教师观**：其身正，不令而行；其身不正，虽令不从。'),
 Document(id='doc_4', metadata={'H1': '第一章 教育概述', 'H2': '第一节 中外教育家及其教育思想', 'H3': '（四）孟子主要思想'}, page_content='### （四）孟子主要思想  \n**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  \n**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  \n**教学原则**：循序渐进，专心有恒。')]

In [27]:
# 带分数查询，结果是list[tuple[Document, float]]
result = vector_store.similarity_search_with_score(
    query="论语中教育的目的是什么",
    k=2
)

for document, score in result:
    print(f"=============={document.id}: {score}===================")
    print(document.model_dump_json(indent=2))

==============doc_3: 0.7214450312388188===================
{
  "id": "doc_3",
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（三）孔子及《论语》主要思想"
  },
  "page_content": "### （三）孔子及《论语》主要思想  \n**教育作用**：庶、富、教；性相近，习相远。  \n**教育对象**：“有教无类”，教育民主思想。  \n**教育目的**：以完善人格为教育的首要目的，培养士和君子。  \n**教学内容**：文、行、忠、义。  \n**教学过程**：学、思、习、行。  \n**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  \n**教师观**：其身正，不令而行；其身不正，虽令不从。",
  "type": "Document"
}
==============doc_4: 0.6494818818743939===================
{
  "id": "doc_4",
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（四）孟子主要思想"
  },
  "page_content": "### （四）孟子主要思想  \n**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  \n**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  \n**教学原则**：循序渐进，专心有恒。",
  "type": "Document"
}


In [37]:
# 在线阶段
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

model = init_chat_model(
    "deepseek-v4-flash",
    extra_body = {"thinking": {"type": "disabled"}}
)

def rag_chat(query: str):
    print("===========用户提问：===========")
    print(query)
    # 1.根据问题检索知识片段
    documents = vector_store.similarity_search(query)

    # 2.把知识片段与问题拼接，形成提示词
    # 2.1.获取知识片段上下文
    context = "\n".join([doc.page_content for doc in documents])
    # 2.2.拼接提示词
    prompt = f"""
    你是一个教资考试专家，你需要根据提供的上下文回答用户的问题，不能自己编造答案。答案要简短，直接引用原文。
    context: ```{context}```
    query: ```{query}```
    """
    print("===========检索文档：===========")
    print(context)

    # 3.调用模型，生成答案
    response = model.invoke(prompt)
    print("===========AI回答：===========")
    print(response.text)

In [38]:
rag_chat("论语中教育的目的是什么？")

===========用户提问：===========
论语中教育的目的是什么？
===========检索文档：===========
### （三）孔子及《论语》主要思想  
**教育作用**：庶、富、教；性相近，习相远。  
**教育对象**：“有教无类”，教育民主思想。  
**教育目的**：以完善人格为教育的首要目的，培养士和君子。  
**教学内容**：文、行、忠、义。  
**教学过程**：学、思、习、行。  
**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  
**教师观**：其身正，不令而行；其身不正，虽令不从。
### （四）孟子主要思想  
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  
**教学原则**：循序渐进，专心有恒。
### （五）荀子主要思想  
**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  
**教学原则**：学以致用，锲而不舍。
### （二）《学记》主要思想  
**教育作用**：化民成俗，其必由学；建国君民，教学为先。  
**教学相长**：教和学两方面互相影响和促进，都得到提高。  
**豫时孙摩**：包括预防性原则、及时性原则、循序渐进原则、集体教育原则。  
**长善救失**：“学者有四失，教者必知之。人之学也，或失则多，或失则寡，或失则易，或失则止。此四者，心之莫同也。知其心，然后能救其失也，教也者，长善而救其失者也。”  
**启发诱导**：道而弗牵，强而弗抑，开而弗达。
===========AI回答：===========
根据上下文，论语中教育目的是“以完善人格为教育的首要目的，培养士和君子”。


In [39]:
rag_chat("你好")

===========用户提问：===========
你好
===========检索文档：===========
### （二）教育的属性  
教育的本质属性是有目的地培养人的社会活动。  
教育的永恒性。  
教育的历史性。  
教育的相对独立性。  
教育的长期性。  
教育的生产性。  
教育的民族性。
## 第二节 教育的定义  
### （一）教育的定义  
广义的教育是指一切有目的地增进人的知识和技能，发展人的智力和体力，影响人的思想品德的社会活动，具有目的性和社会性。广义教育包括社会教育、家庭教育、学校教育。广义的教育是人类社会有史以来就有的教育活动。  
狭义的教育就是指学校教育。  
教育的要素：教育者、受教育者、教育影响（主要是教育内容）。
### （二）经济与教育的关系  
经济发展对教育的影响：经济发展的规模和速度制约着教育发展的规模和速度；经济发展水平制约着教育事业发展的结构和布局；经济发展通过对人才和知识的需求制约着教育目标的设定；经济发展水平制约着学校的教育内容、方式方法手段和教学组织形式；经济发展制约着教育事业发展的基础。  
教育在经济发展中的作用表现为：教育为经济的发展提供良好基础；教育再生产劳动力；教育是科学技术再生产的手段。
## 第二节 教育和社会的关系  
### （一）政治与教育的关系  
政治（经济制度）对教育具有决定作用，是决定教育性质的直接因素。具体表现为：决定了教育的性质和目的；决定了教育的领导权；决定了哪部分社会成员享有受教育的权利；决定了部分的教育内容；决定了教育的管理体制。  
教育对政治发展的作用表现在：教育促进人的政治社会化；教育培养现代政治法律人才；教育促进现代政治民主化。
===========AI回答：===========
您好！请问有什么教资考试方面的问题需要咨询吗？我可以根据您提供的学习材料为您解答。
